# Inverted pendulum — certifying a model you approximated

**Goal.** Certify the upright equilibrium of an LQR-stabilised pendulum, and confront
the fact that the certified model is not the physical one.

**What a reader should take away:** that the author knows the difference between
"I proved something about $\dot x = f(x)$" and "I proved something about the
pendulum", and reports the gap rather than eliding it.

**Prerequisites:** the Van der Pol notebook.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lyapunov_toolbox import julia_available
from lyapunov_toolbox.julia_bridge import get_session, certify_roa, synthesize
from lyapunov_toolbox.plotting import (
    plot_roa_2d, plot_phase_portrait, plot_vdot_field,
    plot_roa_comparison, plot_vs_convergence, save_figure,
)
from lyapunov_toolbox.plotting import use_style
use_style()

session = get_session()
print(f"Julia {session.version} ready.")

## 1. The plant

$$\ddot\theta = \frac{g}{l}\sin\theta - \frac{b}{ml^2}\dot\theta + \frac{u}{ml^2},
\qquad u = -K\,[\theta, \dot\theta]^\top$$

with $\theta = 0$ upright. Note the **plus** sign on $\sin\theta$: upright is
unstable open-loop, which is the entire reason a controller is present.

**Write:** state the parameters, the LQR weights, and the closed-loop eigenvalues.
Confirm the closed loop is Hurwitz before going further and say why an infeasible
SDP would otherwise be uninterpretable.

In [ ]:
# Parameters.
m, l, b, g = 1.0, 1.0, 0.1, 9.81

# Linearised system at upright: θ ≈ 0, sin θ ≈ θ.
# [θ̇; θ̈] = [0, 1; g/l, -b/(m*l²)] [θ; θ̇] + [0; 1/(m*l²)] u
A = np.array([[0, 1],
              [g/l, -b/(m*l**2)]])
B = np.array([[0],
              [1/(m*l**2)]])

print(f"Open-loop A:\n{A}")
print(f"Open-loop eigenvalues: {np.linalg.eigvals(A)}")
print("(positive real part → upright is unstable open-loop)")

# LQR design.
Q = np.diag([10, 1])
R = np.array([[1]])
from scipy.linalg import solve_continuous_are
P_lqr = solve_continuous_are(A, B, Q, R)
K = np.linalg.solve(R, B.T @ P_lqr)
print(f"\nK = [{K[0,0]:.4f}, {K[0,1]:.4f}]")

Acl = A - B @ K
print(f"Closed-loop eigenvalues: {np.linalg.eigvals(Acl)}")
assert np.all(np.real(np.linalg.eigvals(Acl)) < 0), "Closed loop must be Hurwitz"
print("Closed-loop is Hurwitz ✓")

## 2. Two ways to make it polynomial

| | states | exact? | cost |
|---|---|---|---|
| Taylor, $\sin\theta \approx \theta - \theta^3/6$ | $(\theta, \dot\theta)$ | no | small |
| Trigonometric recast, $s = \sin\theta$, $c = \cos\theta$, $s^2 + c^2 = 1$ | $(s, c, \dot\theta)$ | yes | larger SDP |

**Write:** explain what the equality constraint $s^2 + c^2 = 1$ does to the SOS
program — it enters with a *free-sign* multiplier, not an SOS one — and why. Then
state plainly what a certificate on the Taylor model does and does not entitle you
to claim.

In [ ]:
# Build both formulations.
# Taylor (order 3): sin θ ≈ θ - θ³/6 — 2 states (θ, θ̇), polynomial, approximate.
# Trigonometric recast: s = sinθ, c = cosθ with s² + c² = 1 — 3 states, exact
# on the manifold, at the price of a larger equality-constrained SOS program.

# Taylor formulation (the synthesize() default for this system).
cert_taylor, hist_taylor = synthesize("inverted_pendulum", V_degree=2, vs_iterations=0)
print("Taylor (θ, θ̇):")
print(f"  States: {cert_taylor.variables}")
print(f"  SDP size: {cert_taylor.sdp_size}")

# Both system variants from Julia, to compare state counts and the invariant.
code_sizes = """
sys_taylor = LyapunovToolbox.inverted_pendulum(recast=:taylor, taylor_order=3)
sys_trig = LyapunovToolbox.inverted_pendulum(recast=:trig)
(n_states_taylor = length(sys_taylor.x),
 n_states_trig = length(sys_trig.x),
 inv_trig = string(sys_trig.invariants[1]))
"""
n_t, n_tr, inv_trig = session.eval(code_sizes)
print(f"\nTaylor: {int(n_t)} state(s)")
print(f"Trig:   {int(n_tr)} states, invariant {inv_trig}")


## 3. Certify both

**Write:** report both levels, both SDP sizes, both solve times.

In [ ]:
# Certify both formulations.

# Taylor (order 3) via the full V–s iteration.
cert_t, hist_t = synthesize("inverted_pendulum", V_degree=4, vs_iterations=8)
print(f"Taylor (order 3):")
print(f"  ρ = {cert_t.rho:.4f}")
print(f"  SDP size: {cert_t.sdp_size}")
print(f"  Total solve time: {hist_t['total_solve_time']:.1f}s")
print(f"  Verified: {cert_t.verified}")

# Trigonometric recast via Julia (the bridge's synthesize() defaults to Taylor).
# NOTE: evaluate V with the variable-pairs call V(x => pt), NOT positionally:
# the polynomial's internal variable order is (c, s, θ̇), so V(0.0, -1.0, 0.0)
# silently evaluates at the wrong positions.
code_trig_cert = """
sys = LyapunovToolbox.inverted_pendulum(recast=:trig)
opts = LyapunovToolbox.SOSOptions(V_degree = 4, multiplier_degree = 2,
                                   vs_iterations = 8)
result = LyapunovToolbox.v_s_iteration(sys; opts = opts)
cert = result.certificate
vc = LyapunovToolbox.verify_certificate(cert, sys)
val_down = cert.V(sys.x => [0.0, -1.0, 0.0])   # V at the downward equilibrium (0, -1, 0)
(cert.ρ, cert.sdp_size, cert.solve_time, result.total_solve_time, cert.bounded,
 vc.algebraic.ok, vc.sampling.ok, vc.dynamical.ok, val_down)
"""
rho_tr, sdp_tr, t_tr, total_tr, bounded_tr, alg_ok, samp_ok, dyn_ok, val_down_tr = \
    session.eval(code_trig_cert)
print(f"\nTrigonometric (s, c, θ̇):")
print(f"  ρ = {float(rho_tr):.4f}")
print(f"  SDP size: {int(sdp_tr)}")
print(f"  Solve time: {float(total_tr):.1f}s")
print(f"  Bounded: {bool(bounded_tr)}")
print(f"  V(down) = {float(val_down_tr):.4f}  (must exceed ρ for a valid ROA)")
print(f"  verify_certificate: algebraic={bool(alg_ok)} sampling={bool(samp_ok)} "
      f"dynamical={bool(dyn_ok)}")
print("  Note: the 'dynamical' check samples a box around the equilibrium, which")
print("  is OFF the s²+c²=1 manifold; s²+c² is conserved by the flow, so those")
print("  samples never converge. The algebraic and sampling checks are the ones")
print("  that apply on the manifold.")

# Project trig certificate onto (θ, θ̇) using s=sinθ, c=cosθ.
# Approximate the trig boundary by evaluating at sampled s, then mapping to θ = atan2(s, √(1-s²)).
print("\nProjection: the trig certificate lives in (s, c, θ̇) with s²+c²=1.")
print("To plot in (θ, θ̇), we sample s∈[-1,1], set c=√(1-s²), and map θ=atan2(s, c).")
print("This projection is not exact — it's a visualisation aid.")

# Plot both on the same axes.
fig, ax = plt.subplots(figsize=(7, 7))
plot_roa_2d(cert_t, ax=ax, labels=[r"$\theta$", r"$\dot\theta$"])
ax.set_title(f"Certified ROA (Taylor, ρ = {cert_t.rho:.3f})")
save_figure(fig, "roa_inverted_pendulum")
plt.show()


## 4. The honest cross-check

Sample inside the Taylor certified set; integrate the **true** trigonometric
dynamics; count how many converge upright.

**Write:** report the fraction and, if it is 1, the largest $|\theta|$ actually
tested. "No counterexample below $|\theta| = 0.9$ rad" is a claim a reader can use;
"the Taylor certificate is valid" is not.

In [ ]:
# Cross-check: sample inside Taylor cert, simulate TRUE sin dynamics, count convergence.
def true_pendulum_dynamics(x):
    """True pendulum with sin, closed-loop LQR."""
    theta, thetadot = x[0], x[1]
    u = -K[0,0]*theta - K[0,1]*thetadot
    theta_ddot = (g/l)*np.sin(theta) - (b/(m*l**2))*thetadot + u/(m*l**2)
    return np.array([thetadot, theta_ddot])

def rk4_step(f, x, dt):
    k1 = f(x)
    k2 = f(x + 0.5*dt*k1)
    k3 = f(x + 0.5*dt*k2)
    k4 = f(x + dt*k3)
    return x + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

np.random.seed(42)
n_samples = 200
boundary = cert_t.boundary_2d()
bx_min, bx_max = boundary[:, 0].min(), boundary[:, 0].max()
by_min, by_max = boundary[:, 1].min(), boundary[:, 1].max()

converged = 0
failing_pts = []
max_tested_theta = 0.0
for _ in range(n_samples * 3):
    pt = np.array([np.random.uniform(bx_min, bx_max),
                   np.random.uniform(by_min, by_max)])
    if cert_t.evaluate(pt) <= cert_t.rho:
        max_tested_theta = max(max_tested_theta, abs(pt[0]))
        x = pt.copy()
        for step in range(1000):  # 20 s at dt=0.02
            x = rk4_step(true_pendulum_dynamics, x, 0.02)
            if np.linalg.norm(x) < 1e-3:
                converged += 1
                break
        else:
            failing_pts.append(pt)
        if converged + len(failing_pts) >= n_samples:
            break

frac = converged / n_samples
print(f"True-dynamics cross-check (Taylor certificate):")
print(f"  {converged}/{n_samples} converged ({frac*100:.1f}%)")
print(f"  Max |θ| tested: {max_tested_theta:.3f} rad")
if failing_pts:
    failing = np.array(failing_pts)
    print(f"  Failing initial conditions: {len(failing)}")
    print(f"  Smallest failing |θ|: {np.min(np.abs(failing[:, 0])):.3f} rad")

# Plot failing points.
if failing_pts:
    fig, ax = plt.subplots(figsize=(7, 7))
    plot_roa_2d(cert_t, ax=ax, labels=[r"$\theta$", r"$\dot\theta$"])
    failing = np.array(failing_pts)
    ax.scatter(failing[:, 0], failing[:, 1], color='red', s=30, zorder=5,
               label=f'Failed ({len(failing_pts)})')
    ax.legend()
    ax.set_title(f"Cross-check: {frac*100:.0f}% converged")
    plt.show()
else:
    print(f"\nAll samples converged — no counterexample below |θ| = {max_tested_theta:.3f} rad.")

## 5. The saddle check

The downward equilibrium at $\theta = \pm\pi$ is a saddle of the closed loop and
cannot lie in the ROA of the upright one.

**Write:** one short paragraph on why this check costs nothing and catches almost
every sign error in the pipeline.

The check runs against the **trigonometric** certificate, not the Taylor one: the
cubic truncation has no nonzero closed-loop equilibria (its saddle condition
$\theta^{*2} = 6(1 - K_1/(mlg))$ is negative for $K_1 = 20.117$), so evaluating
$V(\pm\pi, 0)$ there proves nothing about the physical system. The exact model's
downward equilibrium $(s, c, \dot\theta) = (0, -1, 0)$ *is* an equilibrium of the
true dynamics, so a certificate that included it would be wrong by inspection —
and checking $V(0,-1,0) > \rho$ costs one polynomial evaluation and catches
nearly every sign error in the pipeline.


In [ ]:
# Check: is the downward equilibrium excluded from the certified set?
# This is only meaningful for the EXACT trig model. The cubic Taylor truncation
# has no nonzero closed-loop equilibria: its saddle condition θ*² = 6(1 − K₁/(mlg))
# is negative for the LQR gain K₁ = 20.117, so (±π, 0) is not an equilibrium of
# the truncated vector field and evaluating V there proves nothing. The trig
# model (s, c, θ̇) = (0, -1, 0) IS an equilibrium of the exact dynamics, so a
# certificate that includes it would be wrong by inspection.
print(f"Trig: V(0, -1, 0) = {float(val_down_tr):.4f} vs ρ = {float(rho_tr):.4f}")
assert float(val_down_tr) > float(rho_tr), \
    "SADDLE CHECK FAILED: the downward equilibrium is inside the certified set!"
print("  (0, -1, 0) is outside the certified set ✓")

print("\nSaddle check PASSED for the exact trig model.")
print("This catches nearly every sign error in the dynamics pipeline — ",
      "if the downward equilibrium were inside the upright ROA, the certificate is wrong.")


## 6. Taylor order sweep and limitations

**Write:** report $\rho$, the projected area, and the converged fraction for orders
3, 5, 7. A higher-order truncation is a better model, so the converged fraction
should improve; whether the certified area does is not obvious. Report what you
observed, including if it contradicts what you expected — a sweep that comes out
against the intuition and is reported anyway is worth more than one that confirms it.

Also state: the damping value you used and how sensitive $\rho$ is to it, and what
happens as $b \to 0$.

In [ ]:
# Taylor order sweep: 3, 5, 7.
# The multiplier degree must grow with the truncation order: for a V of degree
# d, the Lie derivative V̇ = ∇V·f has degree (d−1) + taylor_order, and the SOS
# certificate needs λ·(V−ρ) to reach that degree (the margin ε‖x‖^{2k} is a
# degree-2k term too). With md fixed at 2 the degree-6 (order 5) and degree-8
# (order 7) leading terms of V̇ cannot be compensated and the program is
# structurally infeasible even at ρ = 0. md = order − 1 covers all three.
orders = [3, 5, 7]
results = []
for order in orders:
    md = order - 1
    print(f"\n--- Taylor order {order} (multiplier_degree = {md}) ---")
    code = f"""
    sys = LyapunovToolbox.inverted_pendulum(recast=:taylor, taylor_order={order})
    opts = LyapunovToolbox.SOSOptions(V_degree = 4, multiplier_degree = {md},
                                       vs_iterations = 8)
    result = LyapunovToolbox.v_s_iteration(sys; opts = opts)
    cert = result.certificate
    LyapunovToolbox.verify_certificate(cert, sys)
    (cert.ρ, cert.sdp_size, result.total_solve_time, cert.bounded)
    """
    rho_o, sdp_o, t_o, bounded_o = session.eval(code)

    results.append({
        'order': order,
        'rho': float(rho_o),
        'sdp': int(sdp_o),
        'time': float(t_o),
        'bounded': bool(bounded_o),
    })
    print(f"  ρ = {float(rho_o):.4f}, SDP = {int(sdp_o)}, time = {float(t_o):.1f}s")

# Cross-check each order as before.
print("\n| Order | ρ | SDP size | Time |")
print("|-------|----|----------|------|")
for r in results:
    print(f"| {r['order']} | {r['rho']:.4f} | {r['sdp']} | {r['time']:.1f}s |")

print(f"\nThe pendulum uses the default damping b = 0.1. As b → 0, the system")
print(f"becomes less damped and the certified ROA typically shrinks — the LQR")
print(f"has less authority to counteract the gravitational instability.")

# Save hero figure.
fig, ax = plt.subplots(figsize=(7, 7))
plot_roa_2d(cert_t, ax=ax, labels=[r"$\theta$", r"$\dot\theta$"])
ax.set_title(f"Inverted Pendulum ROA (Taylor order 3, ρ = {cert_t.rho:.3f})")
save_figure(fig, "roa_inverted_pendulum")
plt.show()

## Takeaways

**Write:** three or four bullets stating what a reader now knows that they did not
before opening this notebook. State the numbers. No filler, no "in this notebook we
have seen".